
# SVM Baseline for Malware Detection

## Objectives

The objective of this notebook is to develop a Support Vector Machine (SVM) baseline model for malware detection using the preprocessed BODMAS dataset.

The model classifies Windows executable files into two categories:
- 0: Benign
- 1: Malicious

The main objectives are:
- Load and validate the dataset.
- Analyze the distribution of the target classes.
- Apply feature selection to reduce dimensionality.
- Build an SVM classification model.
- Optimize the model hyperparameters using cross-validation.
- Evaluate the model using classification metrics.
- Save the trained model for future adversarial experiments.


## 1. Data Integration

### 1.1 Loading the Preprocessed Dataset

The preprocessed BODMAS dataset is loaded from the processed-data directory. Data preprocessing and the train/test split have already been performed in an earlier stage of the project.

Each sample is represented by 2,381 numerical features. The dataset is provided in four arrays:

- `X_train` and `y_train`: features and labels used for model training and hyperparameter tuning.
- `X_test` and `y_test`: features and labels reserved for the final evaluation.

The test set is kept separate to ensure an unbiased evaluation of the model's performance.

In [3]:
from pathlib import Path
import numpy as np
import joblib

# Path to the preprocessed dataset
DATA_DIR = Path('../../data/processed/bodmas')

X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')

X_test = np.load(DATA_DIR / 'X_test.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

print("Testing features:", X_test.shape)
print("Testing labels:", y_test.shape)

Training features: (107548, 2381)
Training labels: (107548,)
Testing features: (26887, 2381)
Testing labels: (26887,)


### 1.2 Dataset Integrity and Shape

Before training the machine-learning model, we verify that the loaded arrays are structurally consistent and contain valid numerical values.

The following checks are performed:

- Verify that each feature matrix has the same number of samples as its corresponding label vector.
- Ensure that training and testing datasets have the same number of features.
- Check for missing (`NaN`) and infinite values.
- Verify that the target contains the expected binary labels (0 and 1).
- Examine the distribution of the target classes in both datasets.

These checks help ensure that the data is valid and ready for model training.

In [4]:
# Check dataset dimensions
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)

# Check feature consistency
assert X_train.shape[1] == X_test.shape[1]

# Check missing and infinite values
print("NaN values in training:", np.isnan(X_train).sum())
print("NaN values in testing:", np.isnan(X_test).sum())

print("Infinite values in training:", np.isinf(X_train).sum())
print("Infinite values in testing:", np.isinf(X_test).sum())

# Check label values
print("Training labels:", np.unique(y_train))
print("Testing labels:", np.unique(y_test))

# Check class distribution
print("\nTraining class distribution:")
print(np.unique(y_train, return_counts=True))

print("\nTesting class distribution:")
print(np.unique(y_test, return_counts=True))

# Verify that the data is valid
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

print("\nData validation completed successfully.")

NaN values in training: 0
NaN values in testing: 0
Infinite values in training: 0
Infinite values in testing: 0
Training labels: [0 1]
Testing labels: [0 1]

Training class distribution:
(array([0, 1], dtype=int32), array([61714, 45834]))

Testing class distribution:
(array([0, 1], dtype=int32), array([15428, 11459]))

Data validation completed successfully.


### 1.3 Label Distribution

The distribution of benign and malicious samples is examined
to identify potential class imbalance.

This analysis is important because an imbalanced dataset may
affect the learning process and classification performance.

The class distribution is examined separately for the training
and testing sets to verify the consistency of the data splits.

In [5]:
import pandas as pd

train_distribution = pd.Series(y_train).value_counts().sort_index()
test_distribution = pd.Series(y_test).value_counts().sort_index()

distribution = pd.DataFrame({
    "Training Set": train_distribution,
    "Testing Set": test_distribution
})

distribution.index = ["Benign", "Malicious"]

display(distribution)

,Training Set,Testing Set
Benign,61714,15428
Malicious,45834,11459


## 2. Feature Selection

### 2.1 Feature Selection Strategy

The dataset contains 2,381 features. To reduce computational cost and identify the most informative features, this notebook uses **SelectKBest with the ANOVA F-test (`f_classif`)**.

- `f_classif` evaluates the relationship between each feature and the target classes.
- `SelectKBest` selects the top `k` features based on these scores.

**Why use a Pipeline?**

Feature selection and constant-feature removal are included in the Pipeline to prevent data leakage during cross-validation.

### 2.2 Feature Selection Configuration

Several candidate values of `k` are initially defined, representing the number of features selected by `SelectKBest`.
For the current hyperparameter search, only **100 and 300 features** are considered to reduce computational cost.

The test set is not used during feature selection or hyperparameter tuning.

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif

# Number of available features
n_features = X_train.shape[1]

# Candidate numbers of selected features
k_values = [100, 300, 600, 1000, 'all']

# Keep only valid values
k_values = [
    k for k in k_values
    if k == 'all' or k <= n_features
]

print("Total number of features:", n_features)
print("Feature selection candidates:", k_values)

Total number of features: 2381
Feature selection candidates: [100, 300, 600, 1000, 'all']


### 3. Support Vector Machine (SVM)

This notebook uses **`LinearSVC`**, a linear SVM suitable for high-dimensional datasets. It learns a decision boundary to classify executables as benign or malicious.

The regularization parameter `C` controls the balance between model simplicity and training errors:
- Small `C`: stronger regularization.
- Large `C`: weaker regularization.

The best value of `C` will be selected using cross-validation on the training data.

### 3.2 SVM Pipeline Configuration

The `Pipeline` combines two steps:

1. `VarianceThreshold(threshold=0.0)`: removes constant features that do not provide useful information.
2. `SelectKBest(f_classif)`: selects the most informative features
3. `LinearSVC`: performs the classification.

The classifier uses:
- `class_weight='balanced'`: compensates for class imbalance.
- `max_iter=10000`: allows enough iterations for convergence.
- `random_state=42`: ensures reproducibility where applicable.

The Pipeline also prevents data leakage by performing feature selection separately within each cross-validation fold.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

# Define the SVM pipeline
svm_pipeline = Pipeline([
    ('remove_constant', VarianceThreshold(threshold=0.0)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('svm', LinearSVC(
        class_weight='balanced',
        max_iter=10000,
        random_state=42
    ))
])

print("SVM Pipeline:")
print(svm_pipeline)

SVM Pipeline:
Pipeline(steps=[('remove_constant', VarianceThreshold()),
                ('feature_selection', SelectKBest()),
                ('svm',
                 LinearSVC(class_weight='balanced', max_iter=10000,
                           random_state=42))])


## 4. Model Training and Hyperparameter Tuning

### 4.1 Tuning Strategy

Two hyperparameters are tuned using `GridSearchCV`:

- `feature_selection__k`: number of selected features.
- `svm__C`: regularization parameter of the SVM.

We use **stratified 2-fold cross-validation** to preserve class proportions in each fold.

The **macro-averaged F1-score** is used as the evaluation metric.

Tuning is performed only on the training data (`X_train`, `y_train`). The test set is reserved for final evaluation.


In [18]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Define a lightweight hyperparameter grid
param_grid = {
    'feature_selection__k': [100, 300],
    'svm__C': [0.1, 1, 10]
}

# Define cross-validation strategy
cv_strategy = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

print("Hyperparameter grid:", param_grid)
print("Number of configurations:", 6)

Hyperparameter grid: {'feature_selection__k': [100, 300], 'svm__C': [0.1, 1, 10]}
Number of configurations: 6


### 4.2 GridSearchCV Configuration and Training

`GridSearchCV` combines the Pipeline, the hyperparameter grid and the cross-validation strategy defined above. For each of the 6 configurations, it trains the model on the training folds and scores it on the validation fold.

Key settings:

- `scoring='f1_macro'`: the macro-averaged F1-score is used to rank configurations.
- `n_jobs=1` and `pre_dispatch=1`: models are trained one at a time to limit memory usage.
- `refit=True`: after the search, the best configuration is retrained on the **whole training set**, producing the final model used later for the test evaluation.

The training time is measured to document the computational cost of the tuning.

In [19]:
import time

# Configure GridSearchCV
grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv_strategy,
    n_jobs=1,
    pre_dispatch=1,
    verbose=1,
    refit=True
)

# Start training
start_time = time.time()

grid_search.fit(X_train, y_train)

training_time = time.time() - start_time

print("Training completed successfully!")
print(f"Training time: {training_time:.2f} seconds")

Fitting 2 folds for each of 6 candidates, totalling 12 fits
Training completed successfully!
Training time: 149.56 seconds


### 4.3 Optimal Hyperparameter Selection

After completing the hyperparameter search, the best configuration is selected automatically.

The optimal values of C and k are extracted along with the corresponding cross-validation F1-macro score.

The best estimator is stored for the subsequent evaluation stage.

In [20]:
# Retrieve the best model
best_svm_model = grid_search.best_estimator_

# Display optimal hyperparameters
print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{grid_search.best_score_:.4f}")

Best Hyperparameters:
{'feature_selection__k': 300, 'svm__C': 10}

Best Cross-Validation F1-Macro:
0.9721
